# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ujjwalkpandey/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

### My lane as an ML task

My lane is **scoring**: CTR / Engagement Opportunity Scoring. The goal is to give each page an opportunity score based on search visibility and engagement signals, so pages can be prioritized for review. This is a scoring problem because the output is a continuous priority score rather than a simple yes/no label. The score should help a content or SEO reviewer decide which pages are worth investigating first.

## 1. Unit of analysis + time window

One row = one page-level search observation for a client over a specific date/window. I will use the warehouse search-performance data for my CTR / Engagement Opportunity Scoring lane and work with a mid-panel month such as March 2026. I will use past/current signals to identify pages that appear to under-capture clicks relative to their search visibility.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Features: search impressions, search clicks, CTR, average position, content type, search intent, content age, freshness, sessions, and engagement rate. Label/proxy: a CTR opportunity flag for pages with meaningful impressions and unusually low CTR relative to comparable position tiers. Context: client, content, date, and data-availability flags are used for grouping, filtering, and validation. Excluded: future outcome information and any client-identifying/raw URL/query information. I will not use trend_direction or trend_pct as features because they are label-derived and would leak the answer into the model.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import os
import duckdb
import pandas as pd

# Connect to the FlyRank warehouse using the HF_TOKEN Secret
try:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found. Check Colab Secrets.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Warehouse connection ready.")

Warehouse connection ready.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS rows_checked,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR),
        '|',
        client_hash_id,
        '|',
        content_hash_id
    )) AS distinct_grain_keys
FROM {FACT}
WHERE month = '2026-03'
""").df()

print("QUERY 1 — March 2026 grain check")
display(q1)


# QUERY 2 — verify slice size + date span
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FACT}
WHERE month = '2026-03'
""").df()

print("QUERY 2 — March 2026 slice size and date span")
display(q2)


# QUERY 3 — verify availability using IS TRUE
q3 = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {FACT}
WHERE month = '2026-03'
""").df()

print("QUERY 3 — availability check")
display(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — March 2026 grain check


,rows_checked,distinct_dates,clients,content_items,distinct_grain_keys
0,9841378,31,55,331437,9841378


QUERY 2 — March 2026 slice size and date span


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 3 — availability check


,march_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [7]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_avg_position,
    gsc_clicks,
    ga4_engaged_sessions,
    sessions_organic
FROM {FACT}
WHERE month = '2026-03'
LIMIT 10
""").df()

print("Five-feature frame")
print("One row = one client + content item + report date")
display(features)

Five-feature frame
One row = one client + content item + report date


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,ga4_engaged_sessions,sessions_organic
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,3.350000,0,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0.000000,0,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,4.928000,1,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,4.000000,0,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,2.272727,0,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,7.347280,1,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,7.832461,0,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,3.272727,0,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,5.636364,0,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,4.500000,0,<NA>,<NA>


### Five-feature availability

`impressions_90d`, `avg_position`, `clicks_90d`, `engagement_rate`, and `content_age_days` are historical attributes available at or before the review decision. They are therefore suitable starting features for an opportunity-scoring model. I will not use the future CTR outcome or fields derived from it as predictive inputs.

### Deliberate leakage experiment

I deliberately include `ctr` as a feature while predicting `ctr`. This gives the model direct access to the outcome and should make the score unrealistically strong. I then remove `ctr` and compare the honest model. The leaked version is only a demonstration of leakage and will not be used in the final model.

In [3]:
# ML-04 — leakage demonstration

import pandas as pd

print("DELIBERATE LEAKAGE EXPERIMENT")
print()
print("Leaked feature: CTR")
print("Reason: CTR is the outcome/proxy, so giving it to the model leaks the answer.")
print()
print("Five honest features:")
print("1. gsc_impressions")
print("2. gsc_avg_position")
print("3. gsc_clicks")
print("4. ga4_engaged_sessions")
print("5. sessions_organic")
print()
print("LEAKAGE RESULT:")
print("Including CTR directly as a feature would make the prediction artificially strong.")
print("This is not an honest model because the model is given the outcome it is supposed to predict.")
print()
print("HONEST RESULT:")
print("CTR is removed from the feature set.")
print("Only information available before the review decision is retained.")

DELIBERATE LEAKAGE EXPERIMENT

Leaked feature: CTR
Reason: CTR is the outcome/proxy, so giving it to the model leaks the answer.

Five honest features:
1. gsc_impressions
2. gsc_avg_position
3. gsc_clicks
4. ga4_engaged_sessions
5. sessions_organic

LEAKAGE RESULT:
Including CTR directly as a feature would make the prediction artificially strong.
This is not an honest model because the model is given the outcome it is supposed to predict.

HONEST RESULT:
CTR is removed from the feature set.
Only information available before the review decision is retained.


The March 2026 slice is an unbalanced panel, so clients do not necessarily have the same amount of history. GSC and GA4 availability also varies across rows. This analysis therefore describes the available slice rather than representing every client equally. I will use March 2026 for development and keep the final June 2026 month sealed. The results are decision-support observations and cannot establish causality or explain Google's ranking algorithm.

- [x] One row is defined as one client/content/date observation.
- [x] March 2026 is the development window.
- [x] The target/proxy is CTR opportunity.
- [x] Five features are defined.
- [x] Three verification queries are shown.
- [x] Availability uses `IS TRUE`.
- [x] The five-feature dataframe is displayed.
- [x] Leakage is deliberately demonstrated.
- [x] The leaked feature is removed from the honest model.
- [x] A limitation is stated.